Final RuleFit rule extraction code with no feature duplications or redundant conditions

In [2]:
import os
import json
import numpy as np
import pandas as pd
from rulefit import RuleFit

input_dir = 'zfmis_selected_features_BTTypes/k_3'
train_df = pd.read_csv(os.path.join(input_dir, 'train_selected_features.csv'))
val_df = pd.read_csv(os.path.join(input_dir, 'val_selected_features.csv'))
test_df = pd.read_csv(os.path.join(input_dir, 'test_selected_features.csv'))

train_val_df = pd.concat([train_df, val_df], ignore_index=True)
X_train_val = train_val_df.drop(columns=['label']).values
y_train_val = train_val_df['label'].values
feature_names = train_val_df.columns.drop('label').tolist()

class_names = {0: 'Benign', 1: 'Malignant'}

def format_val(val):
    if abs(val) < 0.01 and val != 0:
        return f'{val:.2e}'
    return f'{val:.4f}'

def simplify_condition_list(cond_list):
    lower_bounds = {}
    upper_bounds = {}
    for feat, op, val in cond_list:
        if op == '>':
            if feat not in lower_bounds or val > lower_bounds[feat]:
                lower_bounds[feat] = val
        elif op == '<=':
            if feat not in upper_bounds or val < upper_bounds[feat]:
                upper_bounds[feat] = val
    clauses = []
    all_feats = sorted(set(list(lower_bounds.keys()) + list(upper_bounds.keys())))
    for f in all_feats:
        has_low = f in lower_bounds
        has_up = f in upper_bounds
        short = 'S' if f == 'entropy' else ('GM' if f == 'geometric_mean' else ('HM' if f == 'harmonic_mean' else f))
        if has_low and has_up:
            clauses.append(f'{short} > {format_val(lower_bounds[f])} & {short} <= {format_val(upper_bounds[f])}')
        elif has_low:
            clauses.append(f'{short} > {format_val(lower_bounds[f])}')
        elif has_up:
            clauses.append(f'{short} <= {format_val(upper_bounds[f])}')
    return ' & '.join(clauses)

with open(os.path.join('rulefit_BTTypes', 'best_rf_params.json'), 'r') as f:
    rf_cfg = json.load(f)
rf_params = rf_cfg['best_rf_params']

rf = RuleFit(random_state=42, **rf_params)
rf.fit(X_train_val, y_train_val, feature_names=feature_names)

rules_df = rf.get_rules()
rules_df = rules_df[rules_df.coef != 0].copy()
rules_df = rules_df.sort_values(by='importance', ascending=False)

rf_formatted_rules = []
for _, row in rules_df.iterrows():
    raw_rule = row['rule']
    coef = row['coef']
    imp = row['importance']
    if raw_rule in feature_names:
        continue
    cond_items = raw_rule.split(' & ')
    cond_parsed = []
    for c in cond_items:
        parts = c.split()
        if len(parts) >= 3:
            f_name, op, val = parts[0], parts[1], float(parts[2])
            cond_parsed.append((f_name, op, val))
    if not cond_parsed:
        continue
    simplified = simplify_condition_list(cond_parsed)
    pred_class = 'Malignant' if coef > 0 else 'Benign'
    rule_str = f'If ({simplified}) then Predicted Class = {pred_class} (Importance: {imp:.4f})'
    rf_formatted_rules.append((rule_str, imp))

rf_formatted_rules = sorted(rf_formatted_rules, key=lambda x: x[1], reverse=True)
rf_rules_file = os.path.join(input_dir, 'simplified_rulefit_top_if_then_rules_for_all_classes.txt')
with open(rf_rules_file, 'w') as f:
    for r, _ in rf_formatted_rules:
        f.write(r + chr(10))

print(f'RuleFit rules saved to: {rf_rules_file}')
print('\nTop Extracted RuleFit Rules (matching Table 13):')
for r, _ in rf_formatted_rules[:6]:
    print(' ', r)


/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.784130e-02, tolerance: 3.400e-02
  model = cd_fast.enet_coordinate_descent(
/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.045176e-02, tolerance: 3.398e-02
  model = cd_fast.enet_coordinate_descent(
/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning

RuleFit rules saved to: zfmis_selected_features_BTTypes/k_3/simplified_rulefit_top_if_then_rules_for_all_classes.txt

Top Extracted RuleFit Rules (matching Table 13):
  If (autocorrelation <= -0.0253 & signal_to_noise > 0.6512) then Predicted Class = Malignant (Importance: 0.1159)
  If (autocorrelation <= 1.07e-03 & S <= 4.0242 & signal_to_noise > 0.6527) then Predicted Class = Malignant (Importance: 0.0867)
  If (autocorrelation <= -0.0221 & signal_to_noise > 0.6517) then Predicted Class = Malignant (Importance: 0.0517)
  If (autocorrelation > -5.49e-03 & autocorrelation <= 0.0471 & signal_to_noise <= 0.7595) then Predicted Class = Malignant (Importance: 0.0413)
  If (autocorrelation <= 0.0239 & signal_to_noise > 0.6543 & signal_to_noise <= 0.7793) then Predicted Class = Malignant (Importance: 0.0318)
  If (S <= 3.9431) then Predicted Class = Malignant (Importance: 0.0317)


Final Decision Tree rule extraction code with no feature duplications or redundant conditions

In [3]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

input_dir = 'zfmis_selected_features_BTTypes/k_3'
train_df = pd.read_csv(os.path.join(input_dir, 'train_selected_features.csv'))
val_df = pd.read_csv(os.path.join(input_dir, 'val_selected_features.csv'))
test_df = pd.read_csv(os.path.join(input_dir, 'test_selected_features.csv'))

train_val_df = pd.concat([train_df, val_df], ignore_index=True)
X_train_val = train_val_df.drop(columns=['label']).values
y_train_val = train_val_df['label'].values
feature_names = train_val_df.columns.drop('label').tolist()

class_names = {0: 'Benign', 1: 'Malignant'}

def format_val(val):
    if abs(val) < 0.01 and val != 0:
        return f'{val:.2e}'
    return f'{val:.4f}'

def simplify_condition_list(cond_list):
    lower_bounds = {}
    upper_bounds = {}
    for feat, op, val in cond_list:
        if op == '>':
            if feat not in lower_bounds or val > lower_bounds[feat]:
                lower_bounds[feat] = val
        elif op == '<=':
            if feat not in upper_bounds or val < upper_bounds[feat]:
                upper_bounds[feat] = val
    clauses = []
    all_feats = sorted(set(list(lower_bounds.keys()) + list(upper_bounds.keys())))
    for f in all_feats:
        has_low = f in lower_bounds
        has_up = f in upper_bounds
        short = 'S' if f == 'entropy' else ('GM' if f == 'geometric_mean' else ('HM' if f == 'harmonic_mean' else f))
        if has_low and has_up:
            clauses.append(f'{short} > {format_val(lower_bounds[f])} & {short} <= {format_val(upper_bounds[f])}')
        elif has_low:
            clauses.append(f'{short} > {format_val(lower_bounds[f])}')
        elif has_up:
            clauses.append(f'{short} <= {format_val(upper_bounds[f])}')
    return ' & '.join(clauses)

with open(os.path.join(input_dir, 'best_dt_params.json'), 'r') as f:
    dt_cfg = json.load(f)
dt_params = dt_cfg['best_dt_params']

clf = DecisionTreeClassifier(random_state=42, **dt_params)
clf.fit(X_train_val, y_train_val)

dt_raw_rules = []
def traverse(tree, node=0, conditions=[], path_importance=0.0):
    if tree.children_left[node] == -1 and tree.children_right[node] == -1:
        class_val = np.argmax(tree.value[node][0])
        dt_raw_rules.append((conditions.copy(), class_val, path_importance))
        return
    feat_idx = tree.feature[node]
    thresh = tree.threshold[node]
    feat_name = feature_names[feat_idx]
    
    traverse(tree, tree.children_left[node], conditions + [(feat_name, '<=', thresh)], path_importance + clf.feature_importances_[feat_idx])
    traverse(tree, tree.children_right[node], conditions + [(feat_name, '>', thresh)], path_importance + clf.feature_importances_[feat_idx])

traverse(clf.tree_)

dt_formatted_rules = []
for conds, class_val, imp in dt_raw_rules:
    simplified = simplify_condition_list(conds)
    rule_str = f'If ({simplified}) then Predicted Class = {class_names.get(class_val, str(class_val))} (Importance: {imp:.4f})'
    dt_formatted_rules.append((rule_str, imp))

dt_sorted_rules = sorted(dt_formatted_rules, key=lambda x: x[1], reverse=True)
dt_rules_file = os.path.join(input_dir, 'simplified_sorted_rules.txt')
with open(dt_rules_file, 'w') as f:
    for r, _ in dt_sorted_rules:
        f.write(r + chr(10))

print(f'Decision Tree rules saved to: {dt_rules_file}')
print('\nTop Extracted Decision Tree Rules (matching Table 12):')
for r, _ in dt_sorted_rules[:6]:
    print(' ', r)


Decision Tree rules saved to: zfmis_selected_features_BTTypes/k_3/simplified_sorted_rules.txt

Top Extracted Decision Tree Rules (matching Table 12):
  If (autocorrelation > 0.0471 & autocorrelation <= 0.0755 & S <= 3.9422 & signal_to_noise <= 0.7626) then Predicted Class = Benign (Importance: 2.6387)
  If (autocorrelation > 0.0471 & autocorrelation <= 0.0755 & S > 3.9422 & signal_to_noise <= 0.7626) then Predicted Class = Benign (Importance: 2.6387)
  If (autocorrelation > 0.0755 & signal_to_noise <= 0.7626) then Predicted Class = Benign (Importance: 2.6109)
  If (autocorrelation > -0.0609 & autocorrelation <= -9.56e-03 & signal_to_noise > 0.7178 & signal_to_noise <= 0.7921) then Predicted Class = Malignant (Importance: 2.0975)
  If (autocorrelation > -0.0609 & autocorrelation <= -9.56e-03 & signal_to_noise > 0.7921) then Predicted Class = Benign (Importance: 2.0975)
  If (autocorrelation <= -0.0609 & S <= 3.8302 & signal_to_noise > 0.6521 & signal_to_noise <= 0.6809) then Predicted C